# 405 — sc-flow-tools Internal Artifact Capture

Runs the **full CTF-C pipeline** on `GSE232025_stereoseq_n256.h5ad` and captures
every intermediate value that sc-flow-tools actually computes internally,
so the results can be compared against ContextFlow's `step_artifacts/`.

The key src change that makes this possible: `_artifact_callback` was added to
`ot_linear_coupling` in `_coupling.py`. It fires from **inside** the function,
after cost_fn runs and after Sinkhorn finishes, giving the real internal state.

Artifact layout mirrors ContextFlow's `step_artifacts/`:
```
405_artifacts/
  02_ot_coupling/first_batch/   # what sc-flow-tools sees on first coupling call
  03_training/                  # loss per step
  05_next_step/t{i}_to_t{i+1}/ # pred.npy, gt.npy, metrics.json per transition
  04_trajectory/                # IVP trajectory [401, n_src, 50]
```

In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
import os
import json
import functools
import numpy as np
import torch
import scanpy as sc
import ot as pot
import warnings

from sc_flow import SCFlow
from sc_flow.backends.torch.coupling import ot_linear_coupling
from sc_flow.data.samplers._train import MultiTransitionSampler
from sc_flow.backends.torch.metrics import EnergyDistance, MaximumMeanDiscrepancy

warnings.filterwarnings("ignore")

In [34]:
# Dataset
DATA_PATH = "400_data/GSE232025_stereoseq_n256.h5ad"
CONDITION_COL = "day"
STAGE_ORDER = ["0", "1", "2", "3", "4"]
STAGE_NAMES = {"0": "Stage44", "1": "Stage54", "2": "Stage57", "3": "Juvenile", "4": "Adult"}
STAGE_TIMES = {"0": 0.0, "1": 0.25, "2": 0.5, "3": 0.75, "4": 1.0}

# CTF-C parameters (matching ContextFlow paper run)
CTF_ALPHA = 0.5
CTF_LAMBDA = 0.5
N_PCA = 50
N_SS = 50  # updated below from actual obsm shape

# Training
N_TRAIN_STEPS = 10_000
LR = 5e-4
HIDDEN_DIM = (100, 100, 100, 100)

# Misc
SEED = 42
N_EVAL = 500
ARTIFACT_DIR = "405_artifacts"

np.random.seed(SEED)
torch.manual_seed(SEED)

In [35]:
adata = sc.read_h5ad(DATA_PATH)
print(f"adata: {adata.shape}")
print(f"obs columns: {list(adata.obs.columns)}")
print(f"obsm keys:   {list(adata.obsm.keys())}")
print(f"\n{CONDITION_COL} distribution:")
print(adata.obs[CONDITION_COL].value_counts().sort_index())

adata: (1280, 10000)
obs columns: ['CellID', 'day', 'cell_id', 'celltype', 'color']
obsm keys:   ['LR_pattern', 'X_pca', 'X_umap', 'local_mean_X_pca', 'spatial']

day distribution:
day
0    256
1    256
2    256
3    256
4    256
Name: count, dtype: int64


In [36]:
for stage in STAGE_ORDER:
    n = (adata.obs[CONDITION_COL] == stage).sum()
    assert n > 0, f"Stage '{stage}' has 0 cells"
    print(f"  day='{stage}' ({STAGE_NAMES[stage]}): {n} cells")

  day='0' (Stage44): 256 cells
  day='1' (Stage54): 256 cells
  day='2' (Stage57): 256 cells
  day='3' (Juvenile): 256 cells
  day='4' (Adult): 256 cells


In [37]:
X_pca = adata.obsm["X_pca"].astype(np.float32)
ss_context = adata.obsm["local_mean_X_pca"].astype(np.float32)
lr_pattern = adata.obsm["LR_pattern"].astype(np.float32)

N_SS = ss_context.shape[1]

adata.obsm["X_state_ctx"] = np.concatenate([X_pca, ss_context, lr_pattern], axis=1)

print(f"X_pca:       {X_pca.shape}")
print(f"ss_context:  {ss_context.shape}")
print(f"lr_pattern:  {lr_pattern.shape}")
print(f"X_state_ctx: {adata.obsm['X_state_ctx'].shape}")
print(f"N_PCA={N_PCA}, N_SS={N_SS}")

X_pca:       (1280, 50)
ss_context:  (1280, 50)
lr_pattern:  (1280, 233)
X_state_ctx: (1280, 333)
N_PCA=50, N_SS=50


In [38]:
adata.uns["stage_emb"] = {s: np.array([t], dtype=np.float32) for s, t in STAGE_TIMES.items()}

MATCHED_KEYS = {
    ("0",): ("1",),
    ("1",): ("2",),
    ("2",): ("3",),
    ("3",): ("4",),
}
TRANSITIONS = [("0", "1"), ("1", "2"), ("2", "3"), ("3", "4")]

SCFlow.register_adata(
    adata,
    sample_rep="X_pca",
    coupling_rep="X_state_ctx",
    conditions={CONDITION_COL: [CONDITION_COL]},
    conditions_reps={CONDITION_COL: "stage_emb"},
    matched_keys=MATCHED_KEYS,
)
print("register_adata done")

register_adata done


## Artifact callback

`_artifact_callback` was added to `ot_linear_coupling` in `_coupling.py`.
It fires from **inside** the function with the actual arrays sc-flow-tools computed:
- `source_lin` / `target_lin`: the PCA vectors as seen by Sinkhorn
- `cost_pre_scale`: output of `cost_fn` before `_scale_distance_matrix`
- `cost_post_scale`: after dividing by mean (`scale_cost='mean'`)
- `coupling_matrix`: Sinkhorn transport plan (`reg=0.5` default)

We only keep the **first** call (first training batch), which is deterministic.

In [ ]:
coupling_artifacts = {}
call_count = [0]


def artifact_callback(d):
    """Capture first coupling call artifacts for debugging."""
    call_count[0] += 1
    if call_count[0] == 1:
        coupling_artifacts.update(d)
        print("[callback] First coupling call captured:")
        for k, v in d.items():
            print(f"  {k}: shape={v.shape}  min={v.min():.4f}  max={v.max():.4f}")

In [40]:
def ctf_c_with_logging(source_lin, target_lin, CTF_LAMBDA, CTF_ALPHA, N_PCA, N_SS, _artifact_callback=None, **kwargs):
    """CTF-C coupling — identical cost to temp.ipynb but passes callback into ot_linear_coupling."""
    src_pca = source_lin[:, :N_PCA]
    src_ss = source_lin[:, N_PCA : N_PCA + N_SS]
    src_lr = source_lin[:, N_PCA + N_SS :]

    tgt_pca = target_lin[:, :N_PCA]
    tgt_ss = target_lin[:, N_PCA : N_PCA + N_SS]
    tgt_lr = target_lin[:, N_PCA + N_SS :]

    def cost_fn(src, tgt):
        C_pca = torch.cdist(src, tgt) ** 2
        C_pca_n = C_pca / C_pca.max()
        SS = torch.cdist(src_ss, tgt_ss) ** 2
        SS_n = SS / SS.max()
        LR = torch.cdist(src_lr, tgt_lr) ** 2
        LR_n = LR / LR.max()
        M = CTF_LAMBDA * SS_n + (1 - CTF_LAMBDA) * LR_n
        return CTF_ALPHA * C_pca_n + (1 - CTF_ALPHA) * M

    # The callback fires inside ot_linear_coupling after cost_fn and after Sinkhorn
    return ot_linear_coupling(
        src_pca,
        tgt_pca,
        cost_fn=cost_fn,
        _artifact_callback=_artifact_callback,
    )


ctf_c_partial = functools.partial(
    ctf_c_with_logging,
    CTF_LAMBDA=CTF_LAMBDA,
    CTF_ALPHA=CTF_ALPHA,
    N_PCA=N_PCA,
    N_SS=N_SS,
    _artifact_callback=artifact_callback,
)
print("ctf_c_partial ready")

ctf_c_partial ready


## Train

The callback fires on the first call to `ot_linear_coupling` during training,
capturing what sc-flow-tools actually feeds into Sinkhorn.

In [41]:
model = SCFlow(
    method_id="cfm",
    backend="torch",
    match_fn=ctf_c_partial,
    vf_decoder_mlp_kwargs={"hidden_dims": HIDDEN_DIM, "activation_cls": torch.nn.SELU},
)
model.train(
    adata,
    callbacks=None,
    sort=True,
    n_train_steps=N_TRAIN_STEPS,
    train_batch_size=256,
    train_sampler=MultiTransitionSampler,
    optim_kwargs={"lr": LR},
    timepoints=[0, 1, 2, 3, 4],
)
print(f"\nTraining done. Total coupling calls: {call_count[0]}")
print(f"Coupling artifacts captured: {list(coupling_artifacts.keys())}")

  0%|          | 0/10000 [00:00<?, ?it/s]

[callback] First coupling call captured:
  source_lin: shape=(256, 50)  min=-15.8771  max=4.5175
  target_lin: shape=(256, 50)  min=-19.5149  max=25.6791
  cost_pre_scale: shape=(256, 256)  min=0.3231  max=0.8290
  cost_post_scale: shape=(256, 256)  min=0.6198  max=1.5902
  coupling_matrix: shape=(256, 256)  min=0.0000  max=0.0000


| loss:2.1754 | step:9999: 100%|██████████| 10000/10000 [12:26<00:00, 13.40it/s] 



Training done. Total coupling calls: 40000
Coupling artifacts captured: ['source_lin', 'target_lin', 'cost_pre_scale', 'cost_post_scale', 'coupling_matrix']


## Save artifacts

In [42]:
# 02_ot_coupling/first_batch/
dir_02 = os.path.join(ARTIFACT_DIR, "02_ot_coupling", "first_batch")
os.makedirs(dir_02, exist_ok=True)

assert coupling_artifacts, "No coupling artifacts captured — check callback wiring"

for key, arr in coupling_artifacts.items():
    out = os.path.join(dir_02, f"{key}.npy")
    np.save(out, arr.astype(np.float32))
    print(f"  {key}: {arr.shape} -> {out}")

meta_02 = {
    "description": "Captured from inside ot_linear_coupling on the first training call",
    "source_lin": "src_pca (X_pca slice of batch) passed to ot_linear_coupling",
    "cost_pre_scale": "cost_fn output before _scale_distance_matrix (already [0,1] normalized by cost_fn)",
    "cost_post_scale": "after dividing by mean (scale_cost='mean')",
    "coupling_matrix": "Sinkhorn output",
    "sinkhorn_reg": 0.5,
    "scale_cost": "mean",
    "ctf_alpha": CTF_ALPHA,
    "ctf_lambda": CTF_LAMBDA,
    "total_coupling_calls_during_training": call_count[0],
}
with open(os.path.join(dir_02, "meta.json"), "w") as f:
    json.dump(meta_02, f, indent=2)
print("Saved meta.json")

  source_lin: (256, 50) -> 405_artifacts\02_ot_coupling\first_batch\source_lin.npy
  target_lin: (256, 50) -> 405_artifacts\02_ot_coupling\first_batch\target_lin.npy
  cost_pre_scale: (256, 256) -> 405_artifacts\02_ot_coupling\first_batch\cost_pre_scale.npy
  cost_post_scale: (256, 256) -> 405_artifacts\02_ot_coupling\first_batch\cost_post_scale.npy
  coupling_matrix: (256, 256) -> 405_artifacts\02_ot_coupling\first_batch\coupling_matrix.npy
Saved meta.json


In [ ]:
# 03_training/
dir_03 = os.path.join(ARTIFACT_DIR, "03_training")
os.makedirs(dir_03, exist_ok=True)

try:
    logs_df = model._trainer.get_train_logs_df()
    losses = logs_df["loss"].values.astype(np.float32)
    np.save(os.path.join(dir_03, "training_losses.npy"), losses)
    with open(os.path.join(dir_03, "training_losses.json"), "w") as f:
        json.dump(losses.tolist(), f)
    print(f"Saved training_losses: {losses.shape}  final={losses[-1]:.4f}")
except Exception as e:  # noqa: BLE001
    print(f"Could not retrieve training logs: {e}")

In [ ]:
def compute_metrics(X_pred, X_gt):
    """Compute W2, MMD, and Energy distance between predicted and ground-truth arrays."""
    rng = np.random.RandomState(SEED)
    n = min(N_EVAL, X_pred.shape[0], X_gt.shape[0])
    Xi = X_pred[rng.choice(X_pred.shape[0], n, replace=False)].astype(np.float64)
    Yi = X_gt[rng.choice(X_gt.shape[0], n, replace=False)].astype(np.float64)

    a, b = pot.unif(n), pot.unif(n)
    M_ot = torch.cdist(torch.tensor(Xi).float(), torch.tensor(Yi).float()).numpy().astype(np.float64) ** 2
    w2 = float(np.sqrt(pot.emd2(a, b, M_ot)))

    mmd_m = MaximumMeanDiscrepancy()
    mmd_m.update(torch.tensor(Xi).float(), torch.tensor(Yi).float())
    mmd = float(mmd_m.compute())

    e_m = EnergyDistance()
    e_m.update(torch.tensor(Xi).float(), torch.tensor(Yi).float())
    energy = float(e_m.compute())

    return {"wasserstein": w2, "mmd": mmd, "energy": energy}

In [45]:
# 05_next_step/
dir_05 = os.path.join(ARTIFACT_DIR, "05_next_step")
summary_metrics = {}

for i, (src, tgt) in enumerate(TRANSITIONS):
    label = f"t{i}_to_t{i + 1}"
    print(f"Predicting {src}->{tgt} ({label})...")

    result = model.predict(
        adata,
        sort=True,
        matched_keys={(src,): (tgt,)},
        num_steps=400,
        t_start=i,
        t_end=i + 1,
    )
    pred = np.array(result.X).astype(np.float32)
    gt = adata[adata.obs[CONDITION_COL] == tgt].obsm["X_pca"].astype(np.float32)
    metrics = compute_metrics(pred, gt)
    summary_metrics[label] = metrics

    out_dir = os.path.join(dir_05, label)
    os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, "pred.npy"), pred)
    np.save(os.path.join(out_dir, "gt.npy"), gt)
    with open(os.path.join(out_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(
        f"  pred={pred.shape}  gt={gt.shape}  "
        f"W2={metrics['wasserstein']:.4f}  MMD={metrics['mmd']:.4f}  Energy={metrics['energy']:.4f}"
    )

with open(os.path.join(dir_05, "summary_metrics.json"), "w") as f:
    json.dump(summary_metrics, f, indent=2)
print("Saved 05_next_step summary")

Predicting 0->1 (t0_to_t1)...


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.01it/s]


  pred=(256, 50)  gt=(256, 50)  W2=9.2275  MMD=0.0205  Energy=1.1366
Predicting 1->2 (t1_to_t2)...


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]


  pred=(256, 50)  gt=(256, 50)  W2=8.5325  MMD=0.0089  Energy=0.2170
Predicting 2->3 (t2_to_t3)...


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


  pred=(256, 50)  gt=(256, 50)  W2=6.2116  MMD=0.0137  Energy=0.3103
Predicting 3->4 (t3_to_t4)...


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.22it/s]

  pred=(256, 50)  gt=(256, 50)  W2=6.4114  MMD=0.0143  Energy=0.3765
Saved 05_next_step summary


In [46]:
# 04_trajectory/ — IVP from day '0' to day '4'
dir_04 = os.path.join(ARTIFACT_DIR, "04_trajectory")
os.makedirs(dir_04, exist_ok=True)

print("Running IVP (t=0->4, 401 steps)...")
pred_adata, pred_data = model.predict(
    adata,
    sort=True,
    matched_keys={("0",): ("4",)},
    return_trajectory=True,
    return_tensors=True,
    num_steps=401,
    t_start=0,
    t_end=4,
)
traj = pred_data.traj.detach().cpu().numpy().astype(np.float32)
print(f"traj.shape: {traj.shape}  (n_steps, n_src_cells, n_pca)")

np.save(os.path.join(dir_04, "trajectory.npy"), traj)
print("Saved trajectory.npy")

Running IVP (t=0->4, 401 steps)...


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.00it/s]

traj.shape: (401, 256, 50)  (n_steps, n_src_cells, n_pca)
Saved trajectory.npy


In [47]:
print("=" * 60)
print(f"Artifacts: {os.path.abspath(ARTIFACT_DIR)}")
print()
print("02_ot_coupling/first_batch/")
for k, v in coupling_artifacts.items():
    print(f"  {k}.npy  {v.shape}")
print()
print("05_next_step/ summary:")
for label, m in summary_metrics.items():
    print(f"  {label}: W2={m['wasserstein']:.4f}  MMD={m['mmd']:.4f}  Energy={m['energy']:.4f}")
print()
print(f"04_trajectory/trajectory.npy  {traj.shape}")
print("=" * 60)

Artifacts: c:\Users\HP\Documents\munich_work\sc-flow-tools\docs\notebooks\examples\tutorials\405_artifacts

02_ot_coupling/first_batch/
  source_lin.npy  (256, 50)
  target_lin.npy  (256, 50)
  cost_pre_scale.npy  (256, 256)
  cost_post_scale.npy  (256, 256)
  coupling_matrix.npy  (256, 256)

05_next_step/ summary:
  t0_to_t1: W2=9.2275  MMD=0.0205  Energy=1.1366
  t1_to_t2: W2=8.5325  MMD=0.0089  Energy=0.2170
  t2_to_t3: W2=6.2116  MMD=0.0137  Energy=0.3103
  t3_to_t4: W2=6.4114  MMD=0.0143  Energy=0.3765

04_trajectory/trajectory.npy  (401, 256, 50)


In [51]:
import os

os.getcwd()

'c:\\Users\\HP\\Documents\\munich_work\\sc-flow-tools\\docs\\notebooks\\examples\\tutorials'

In [52]:
gd_t01_loc = r"405_artifacts\02_ot_coupling\t0_to_t1\gene_dist_n.npy"
gd_t01 = np.load(gd_t01_loc)

In [54]:
gd_t01

array([[0.5667846 , 0.41337377, 0.4949787 , ..., 0.4589833 , 0.71346027,
        0.75411564],
       [0.57242125, 0.41550034, 0.49295264, ..., 0.4631735 , 0.7014065 ,
        0.7690912 ],
       [0.5818922 , 0.4120174 , 0.46248743, ..., 0.49005076, 0.6921523 ,
        0.81600946],
       ...,
       [0.54005677, 0.41699928, 0.505089  , ..., 0.44354323, 0.66842955,
        0.70460266],
       [0.5428789 , 0.38502496, 0.45107138, ..., 0.4438077 , 0.62031454,
        0.767312  ],
       [0.47260347, 0.326249  , 0.40387362, ..., 0.40880817, 0.49766645,
        0.71036303]], shape=(256, 256), dtype=float32)